# Tajweed Analysis Model Training on Kaggle TPU

**Architecture**: Fine-tuned Wav2Vec2 XLS-R 300M for 6-class Tajweed classification

**Classes**:
- 0: Correct (no error)
- 1: Al-Madd (elongation)
- 2: Ghunnah (nasalization)
- 3: Ikhfāʾ (concealment)
- 4: Idghām (assimilation)
- 5: Qalqalah (echoing)

**Target**: 93-95% accuracy in 10-15 min on TPU v5e-8

**Export**: TFLite for low-latency real-time inference

## 1. Setup & TPU Initialization

In [ ]:
# Install required packages
!pip install -q transformers[tf] datasets librosa soundfile tf-models-official tensorflowjs

In [ ]:
import os
import numpy as np
import tensorflow as tf
import librosa
from transformers import Wav2Vec2FeatureExtractor, TFWav2Vec2Model
from datasets import load_dataset, Audio
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import json

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

In [ ]:
# TPU Strategy Setup
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver.connect()
    print("Device:", tpu.master())
    strategy = tf.distribute.TPUStrategy(tpu)
    print(f"Number of replicas: {strategy.num_replicas_in_sync}")
except ValueError:
    print("TPU not found, using default strategy (GPU/CPU)")
    strategy = tf.distribute.get_strategy()

print(f"Using strategy: {type(strategy).__name__}")
print(f"Num replicas: {strategy.num_replicas_in_sync}")

## 2. Configuration

In [ ]:
# Training Configuration
CONFIG = {
    # Model
    'pretrained_model': 'facebook/wav2vec2-xls-r-300m',
    'freeze_layers': -4,  # Freeze all except last 4 transformer blocks
    'num_classes': 6,
    
    # Audio
    'sample_rate': 16000,
    'max_duration': 10,  # seconds
    'max_length': 16000 * 10,  # 10 seconds at 16kHz
    
    # Training
    'batch_size': 32,
    'epochs': 10,
    'learning_rate': 2e-5,
    'warmup_steps': 100,
    'label_smoothing': 0.1,
    'weight_decay': 0.01,
    
    # Augmentation
    'augment_prob': 0.5,
    'noise_level': 0.005,
    'speed_range': (0.9, 1.1),
    'spec_augment': True,
    
    # Paths
    'output_dir': '/kaggle/working',
    'model_save_path': '/kaggle/working/tajweed_model.keras',
    'tflite_path': '/kaggle/working/tajweed_model.tflite',
}

# Enable mixed precision for TPU
tf.keras.mixed_precision.set_global_policy('mixed_bfloat16')

# Enable XLA
tf.config.optimizer.set_jit(True)

print(json.dumps(CONFIG, indent=2))

## 3. Dataset Loading & Preprocessing

In [ ]:
# Load QDAT dataset
# Note: Replace with actual dataset path on Kaggle
# For this example, we'll simulate the dataset structure

# OPTION 1: If QDAT is uploaded to Kaggle dataset:
# ds = load_dataset('path/to/qdat', split='train')

# OPTION 2: Load from local files
def load_qdat_dataset():
    """
    Load QDAT dataset for Al-Ma'idah verse 109.
    Expected structure:
    /kaggle/input/qdat/
        └── al-maidah-109/
            ├── correct/
            ├── madd/
            ├── ghunnah/
            ├── ikhfa/
            ├── idgham/
            └── qalqalah/
    """
    import glob
    
    data = []
    labels = []
    
    class_names = ['correct', 'madd', 'ghunnah', 'ikhfa', 'idgham', 'qalqalah']
    label_map = {name: idx for idx, name in enumerate(class_names)}
    
    base_path = '/kaggle/input/qdat/al-maidah-109'
    
    for class_name in class_names:
        class_path = os.path.join(base_path, class_name)
        audio_files = glob.glob(os.path.join(class_path, '*.wav'))
        
        print(f"{class_name}: {len(audio_files)} samples")
        
        for audio_file in audio_files:
            data.append(audio_file)
            labels.append(label_map[class_name])
    
    return np.array(data), np.array(labels), class_names

# Load dataset
try:
    audio_paths, labels, class_names = load_qdat_dataset()
    print(f"\nTotal samples: {len(audio_paths)}")
    print(f"Class distribution: {np.bincount(labels)}")
except FileNotFoundError:
    print("QDAT dataset not found. Using dummy data for demonstration.")
    # Create dummy data for testing
    audio_paths = np.array([f'dummy_{i}.wav' for i in range(1500)])
    labels = np.random.randint(0, 6, 1500)
    class_names = ['correct', 'madd', 'ghunnah', 'ikhfa', 'idgham', 'qalqalah']

In [ ]:
# Train/Val/Test split
X_temp, X_test, y_temp, y_test = train_test_split(
    audio_paths, labels, test_size=0.15, stratify=labels, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15, stratify=y_temp, random_state=42
)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"Train distribution: {np.bincount(y_train)}")
print(f"Val distribution: {np.bincount(y_val)}")
print(f"Test distribution: {np.bincount(y_test)}")

## 4. Data Augmentation

In [ ]:
@tf.function
def add_noise(audio, noise_level=0.005):
    """Add random Gaussian noise."""
    noise = tf.random.normal(tf.shape(audio), stddev=noise_level)
    return audio + noise

@tf.function
def time_stretch(audio, rate):
    """Time stretching (speed change)."""
    # Simple resampling for speed change
    length = tf.cast(tf.shape(audio)[0], tf.float32)
    new_length = tf.cast(length / rate, tf.int32)
    indices = tf.linspace(0.0, length - 1, new_length)
    return tf.gather(audio, tf.cast(indices, tf.int32))

def augment_audio(audio, sample_rate=16000, augment_prob=0.5):
    """
    Apply random augmentations.
    Note: SpecAugment is applied in the feature extractor.
    """
    if np.random.random() < augment_prob:
        # Add noise
        if np.random.random() < 0.5:
            audio = add_noise(audio, noise_level=CONFIG['noise_level'])
        
        # Time stretch
        if np.random.random() < 0.5:
            rate = np.random.uniform(*CONFIG['speed_range'])
            audio = time_stretch(audio, rate)
    
    return audio

def load_and_preprocess_audio(file_path, label, augment=False):
    """
    Load audio file and preprocess.
    """
    # Load audio
    audio, sr = librosa.load(file_path, sr=CONFIG['sample_rate'])
    
    # Convert to tensor
    audio = tf.constant(audio, dtype=tf.float32)
    
    # Augment if training
    if augment:
        audio = augment_audio(audio)
    
    # Pad or truncate to max_length
    max_len = CONFIG['max_length']
    if len(audio) < max_len:
        padding = max_len - len(audio)
        audio = tf.concat([audio, tf.zeros(padding)], axis=0)
    else:
        audio = audio[:max_len]
    
    return audio, label

## 5. Feature Extraction with Wav2Vec2

In [ ]:
# Initialize Wav2Vec2 feature extractor
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    CONFIG['pretrained_model']
)

print(f"Feature extractor sampling rate: {feature_extractor.sampling_rate}")
print(f"Feature size: {feature_extractor.feature_size}")

In [ ]:
def create_tf_dataset(audio_paths, labels, batch_size, shuffle=True, augment=False):
    """
    Create TensorFlow dataset optimized for TPU.
    """
    def generator():
        indices = np.arange(len(audio_paths))
        if shuffle:
            np.random.shuffle(indices)
        
        for idx in indices:
            try:
                audio, label = load_and_preprocess_audio(
                    audio_paths[idx], 
                    labels[idx],
                    augment=augment
                )
                
                # Extract features using Wav2Vec2
                inputs = feature_extractor(
                    audio.numpy(),
                    sampling_rate=CONFIG['sample_rate'],
                    return_tensors="np",
                    padding=True
                )
                
                yield inputs['input_values'][0], label
            except Exception as e:
                print(f"Error loading {audio_paths[idx]}: {e}")
                continue
    
    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(None,), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.int32)
        )
    )
    
    # TPU optimization
    dataset = dataset.batch(batch_size, drop_remainder=True)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset

# Create datasets
print("Creating datasets...")
train_dataset = create_tf_dataset(
    X_train, y_train, 
    CONFIG['batch_size'], 
    shuffle=True, 
    augment=True
)

val_dataset = create_tf_dataset(
    X_val, y_val, 
    CONFIG['batch_size'], 
    shuffle=False, 
    augment=False
)

print("Datasets created!")

## 6. Model Architecture

In [ ]:
def build_tajweed_model(config):
    """
    Build fine-tuned Wav2Vec2 model for Tajweed classification.
    
    Architecture:
    - Wav2Vec2 XLS-R 300M (freeze all except last 4 blocks)
    - Conv1D(256 → 128)
    - Bidirectional LSTM(128)
    - GlobalAveragePooling1D
    - Dense(6, softmax)
    """
    with strategy.scope():
        # Load pre-trained Wav2Vec2
        wav2vec2 = TFWav2Vec2Model.from_pretrained(
            config['pretrained_model'],
            from_pt=True
        )
        
        # Freeze all layers except last 4 transformer blocks
        for layer in wav2vec2.layers[:-4]:
            layer.trainable = False
        
        # Input
        input_values = tf.keras.Input(
            shape=(None,), 
            dtype=tf.float32, 
            name='input_values'
        )
        
        # Wav2Vec2 encoder
        hidden_states = wav2vec2(input_values).last_hidden_state
        
        # Classification head
        x = tf.keras.layers.Conv1D(
            256, 
            kernel_size=3, 
            padding='same',
            activation='relu',
            name='conv1d_256'
        )(hidden_states)
        
        x = tf.keras.layers.Conv1D(
            128,
            kernel_size=3,
            padding='same', 
            activation='relu',
            name='conv1d_128'
        )(x)
        
        x = tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(
                128,
                return_sequences=True,
                name='lstm_128'
            ),
            name='bidirectional_lstm'
        )(x)
        
        x = tf.keras.layers.GlobalAveragePooling1D(name='global_avg_pool')(x)
        
        x = tf.keras.layers.Dropout(0.3, name='dropout')(x)
        
        outputs = tf.keras.layers.Dense(
            config['num_classes'],
            activation='softmax',
            dtype='float32',  # Ensure float32 for final layer
            name='classifier'
        )(x)
        
        model = tf.keras.Model(inputs=input_values, outputs=outputs)
        
        return model

# Build model
print("Building model...")
model = build_tajweed_model(CONFIG)
model.summary()

In [ ]:
# Count trainable parameters
trainable_params = sum([tf.size(var).numpy() for var in model.trainable_variables])
total_params = sum([tf.size(var).numpy() for var in model.variables])

print(f"\nTrainable parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")
print(f"Percentage trainable: {100 * trainable_params / total_params:.2f}%")

## 7. Training Setup

In [ ]:
with strategy.scope():
    # Learning rate schedule with warmup
    def create_lr_schedule(initial_lr, warmup_steps, total_steps):
        def lr_schedule(step):
            if step < warmup_steps:
                return initial_lr * (step / warmup_steps)
            else:
                decay = (total_steps - step) / (total_steps - warmup_steps)
                return initial_lr * decay
        return lr_schedule
    
    steps_per_epoch = len(X_train) // CONFIG['batch_size']
    total_steps = steps_per_epoch * CONFIG['epochs']
    
    lr_schedule = tf.keras.optimizers.schedules.LearningRateSchedule(
        create_lr_schedule(
            CONFIG['learning_rate'],
            CONFIG['warmup_steps'],
            total_steps
        )
    )
    
    # AdamW optimizer
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=CONFIG['learning_rate'],
        weight_decay=CONFIG['weight_decay']
    )
    
    # Loss with label smoothing
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=False,
        label_smoothing=CONFIG['label_smoothing']
    )
    
    # Compile model
    model.compile(
        optimizer=optimizer,
        loss=loss_fn,
        metrics=[
            tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy'),
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top3_accuracy')
        ],
        jit_compile=True  # Enable XLA compilation
    )

print("Model compiled with XLA optimization!")

In [ ]:
# Callbacks
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        CONFIG['model_save_path'],
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=3,
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.TensorBoard(
        log_dir=os.path.join(CONFIG['output_dir'], 'logs'),
        histogram_freq=1
    )
]

## 8. Training

In [ ]:
import time

print("Starting training...")
start_time = time.time()

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=CONFIG['epochs'],
    callbacks=callbacks,
    verbose=1
)

training_time = time.time() - start_time
print(f"\n✅ Training completed in {training_time/60:.2f} minutes")

## 9. Evaluation

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss
axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'training_history.png'), dpi=150)
plt.show()

# Print best metrics
best_epoch = np.argmax(history.history['val_accuracy'])
print(f"\nBest Epoch: {best_epoch + 1}")
print(f"Best Val Accuracy: {history.history['val_accuracy'][best_epoch]:.4f}")
print(f"Best Val Loss: {history.history['val_loss'][best_epoch]:.4f}")

In [ ]:
# Evaluate on test set
test_dataset = create_tf_dataset(
    X_test, y_test,
    CONFIG['batch_size'],
    shuffle=False,
    augment=False
)

print("Evaluating on test set...")
test_loss, test_acc, test_top3_acc = model.evaluate(test_dataset, verbose=1)

print(f"\n📊 Test Results:")
print(f"  Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"  Top-3 Accuracy: {test_top3_acc:.4f}")
print(f"  Loss: {test_loss:.4f}")

In [ ]:
# Confusion Matrix
y_pred = []
y_true = []

for batch_x, batch_y in test_dataset:
    predictions = model.predict(batch_x, verbose=0)
    y_pred.extend(np.argmax(predictions, axis=1))
    y_true.extend(batch_y.numpy())

y_pred = np.array(y_pred)
y_true = np.array(y_true)

# Classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'confusion_matrix.png'), dpi=150)
plt.show()

## 10. Export for Production

In [ ]:
# Save Keras model
print("Saving Keras model...")
model.save(CONFIG['model_save_path'], save_format='keras')
print(f"✅ Saved to {CONFIG['model_save_path']}")

In [ ]:
# Convert to TFLite for low-latency inference
print("\nConverting to TFLite...")

converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Optimizations for speed
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]  # Use FP16 for faster inference

# Convert
tflite_model = converter.convert()

# Save
with open(CONFIG['tflite_path'], 'wb') as f:
    f.write(tflite_model)

tflite_size = len(tflite_model) / 1024 / 1024
print(f"✅ TFLite model saved: {CONFIG['tflite_path']}")
print(f"   Size: {tflite_size:.2f} MB")

In [ ]:
# Test TFLite inference speed
import time

interpreter = tf.lite.Interpreter(model_path=CONFIG['tflite_path'])
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("\nTFLite Inference Speed Test:")
print(f"Input shape: {input_details[0]['shape']}")
print(f"Output shape: {output_details[0]['shape']}")

# Create dummy input
dummy_input = np.random.randn(1, CONFIG['max_length']).astype(np.float32)

# Warmup
for _ in range(10):
    interpreter.set_tensor(input_details[0]['index'], dummy_input)
    interpreter.invoke()

# Measure
num_runs = 100
start_time = time.time()
for _ in range(num_runs):
    interpreter.set_tensor(input_details[0]['index'], dummy_input)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])
elapsed = time.time() - start_time

avg_latency = (elapsed / num_runs) * 1000
fps = num_runs / elapsed

print(f"\n⚡ Performance:")
print(f"   Average latency: {avg_latency:.2f} ms")
print(f"   Throughput: {fps:.2f} inferences/sec")
print(f"   {'✅ Real-time capable!' if avg_latency < 100 else '⚠️ May be slow for real-time'}")

## 11. Save Configuration & Metadata

In [ ]:
# Save model metadata
metadata = {
    'model_name': 'Tajweed Wav2Vec2 Classifier',
    'version': '1.0',
    'architecture': 'facebook/wav2vec2-xls-r-300m + Custom Head',
    'num_classes': CONFIG['num_classes'],
    'class_names': class_names,
    'sample_rate': CONFIG['sample_rate'],
    'max_duration': CONFIG['max_duration'],
    'training': {
        'epochs': CONFIG['epochs'],
        'batch_size': CONFIG['batch_size'],
        'learning_rate': CONFIG['learning_rate'],
        'train_samples': len(X_train),
        'val_samples': len(X_val),
        'test_samples': len(X_test),
        'training_time_minutes': round(training_time / 60, 2)
    },
    'performance': {
        'test_accuracy': float(test_acc),
        'test_loss': float(test_loss),
        'best_val_accuracy': float(history.history['val_accuracy'][best_epoch]),
        'tflite_latency_ms': round(avg_latency, 2),
        'tflite_size_mb': round(tflite_size, 2)
    },
    'created_at': time.strftime('%Y-%m-%d %H:%M:%S')
}

metadata_path = os.path.join(CONFIG['output_dir'], 'model_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print("\n📋 Model Metadata:")
print(json.dumps(metadata, indent=2))
print(f"\n✅ Saved to {metadata_path}")

## 12. Summary & Next Steps

In [ ]:
print("\n" + "="*70)
print("  TRAINING COMPLETE - TAJWEED WAV2VEC2 MODEL")
print("="*70)
print(f"\n📊 Final Results:")
print(f"   Test Accuracy: {test_acc*100:.2f}%")
print(f"   Training Time: {training_time/60:.2f} minutes")
print(f"   TFLite Latency: {avg_latency:.2f} ms")
print(f"\n📦 Saved Artifacts:")
print(f"   - Keras Model: {CONFIG['model_save_path']}")
print(f"   - TFLite Model: {CONFIG['tflite_path']}")
print(f"   - Metadata: {metadata_path}")
print(f"   - Training Plot: training_history.png")
print(f"   - Confusion Matrix: confusion_matrix.png")
print(f"\n🚀 Next Steps:")
print(f"   1. Download models from /kaggle/working/")
print(f"   2. Copy to backend-flask/models/")
print(f"   3. Update MODEL_PATH in .env")
print(f"   4. Test with real audio samples")
print(f"   5. Deploy to production")
print("\n" + "="*70)